# 01. Conversational Core

Covers **Attributes 1, 2, 3, 4, 8, 17**:
- Single & multi-turn dialogs
- Greetings, capability intro, and out-of-scope handling
- Intent validation before data retrieval
- Clarification for ambiguous requests
- Context preservation across turns
- Long-running session memory optimization

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1] if "high_level" in str(pathlib.Path.cwd()) or "capabilities" in str(pathlib.Path.cwd()) else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.orchestrator import Orchestrator
from src.llm_client import MockLLMClient
from src.tools.sql_tool import run_query, validate_sql
from src.tools.retrieval_tool import get_index
from src.tools.code_tool import run_code
from src.formatting import format_value, rows_to_markdown_table

print("AB InBev Enterprise Q&A Agent Pipeline Loaded.")

AB InBev Enterprise Q&A Agent Pipeline Loaded.

### 1. Greetings, Capability Introduction & Out-of-Scope Requests

In [2]:
orch = Orchestrator(llm_router=MockLLMClient(), llm_worker=MockLLMClient())
for q in ["Hello!", "What can you do?", "What is the weather in London today?"]:
    r = orch.handle_turn(q)
    print(f"User: {q}\nIntent: {r.intent}\nAnswer: {r.answer[:120]}...\n")

User: Hello!
Intent: greeting
Answer: Hello! I'm the Anheuser-Busch InBev Q&A assistant. Anheuser-Busch InBev (AB InBev) global performance across its Premium...

User: What can you do?
Intent: capability_intro
Answer: I'm the Anheuser-Busch InBev Q&A assistant. I can:
- Answer questions about Net Revenue, Volume, Market Share, Net Reven...

User: What is the weather in London today?
Intent: out_of_scope
Answer: That's outside what I can help with -- I'm scoped to Anheuser-Busch InBev's business data and related market/company con...


### 2. Intent Validation & Ambiguity Clarification

In [3]:
for q in ["Can you tell me about the performance?", "What was Budweiser revenue in US in 2025?"]:
    r = orch.handle_turn(q)
    print(f"User: {q}\nIntent: {r.intent}\nNeeds Clarification: {r.intent == 'clarification_needed'}\nSub-Agents Used: {r.sub_agents_used}\nAnswer: {r.answer[:140]}...\n")

User: Can you tell me about the performance?
Intent: clarification_needed
Needs Clarification: True
Sub-Agents Used: []
Answer: Could you please clarify which brand (e.g. Corona, Budweiser, Michelob ULTRA), country, or time period you'd like performance details for?...

User: What was Budweiser revenue in US in 2025?
Intent: data_query
Needs Clarification: False
Sub-Agents Used: ['structured']
Answer: In 2025, Budweiser in United States recorded net revenue of **$3,483,878** and volume of **57,558.2 hL**.

| Brand | Country | Year | Net Re...


### 3. Multi-Turn Context Preservation & Filter Persistence

In [4]:
orch = Orchestrator(llm_router=MockLLMClient(), llm_worker=MockLLMClient())
turns = [
    "What was Budweiser revenue in the United States in 2025?",
    "What about in 2024?",
    "And what was the volume in that year?"
]
for t in turns:
    r = orch.handle_turn(t)
    print(f">> {t}")
    print(f"Active Filters: {orch.memory.active_filters}")
    print(f"SQL Used: {r.sql_used}\n")

>> What was Budweiser revenue in the United States in 2025?
Active Filters: {'brand': 'Budweiser', 'country': 'United States', 'kpi': 'net_revenue_usd', 'period': '2025'}
SQL Used: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi WHERE brand='Budweiser' AND country='United States' AND year=2025 GROUP BY brand, country, year LIMIT 500

>> What about in 2024?
Active Filters: {'brand': 'Budweiser', 'country': 'United States', 'kpi': 'net_revenue_usd', 'period': '2024'}
SQL Used: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi WHERE brand='Budweiser' AND country='United States' AND year=2024 GROUP BY brand, country, year LIMIT 500

>> And what was the volume in that year?
Active Filters: {'brand': 'Budweiser', 'country': 'United States', 'kpi': 'volume', 'period': '2024'}
SQL 

### 4. Memory Optimization (Rolling Summarization for Long Sessions)

In [5]:
print(f"Initial raw turns: {len(orch.memory.raw_turns)}")
for i in range(12):
    orch.handle_turn(f"Follow-up step {i} regarding performance")
print(f"Final raw turns bounded: {len(orch.memory.raw_turns)}")
print(f"Rolling summary generated:\n{orch.memory.rolling_summary}")

Initial raw turns: 6
Final raw turns bounded: 10
Rolling summary generated:
[mock-llm output]